In [0]:
#Notebook 02: Reading from Staging and writing to Raw

from datetime import datetime

#Widgets
dbutils.widgets.text("catalog_name", "chinook_catalog")
dbutils.widgets.text("source_catalog", "chinook_sqlserver_team7_catalog")
dbutils.widgets.text("source_schema", "chinook")
dbutils.widgets.text("meta_schema", "pipeline_metadata")
dbutils.widgets.text("staging_schema", "staging")
dbutils.widgets.text("base_path", "/Volumes/chinook_catalog/raw_zone/chinook")

dbutils.widgets.remove("base_path")
dbutils.widgets.text("base_path", "/Volumes/chinook_catalog/raw_zone/chinook")
base_path = dbutils.widgets.get("base_path")
print(repr(base_path))

catalog_name    = dbutils.widgets.get("catalog_name")
source_catalog  = dbutils.widgets.get("source_catalog")
source_schema   = dbutils.widgets.get("source_schema")
meta_schema     = dbutils.widgets.get("meta_schema")
staging_schema  = dbutils.widgets.get("staging_schema")
base_path       = dbutils.widgets.get("base_path")

#Read active tables from metadata
active_tables_df = spark.sql(f"""
    SELECT table_name 
    FROM {catalog_name}.{meta_schema}.parent_metadata
    WHERE active_flag = 'Y'
""")

active_tables = [row["table_name"] for row in active_tables_df.collect()]


#Looping through each table
for table_name in active_tables:
    try:
        print(repr(base_path))
        print(repr(file_location) if 'file_location' in dir() else "file_location not yet defined")
        #Step 1: Read from staging
        staging_df = spark.sql(f"""
            SELECT * FROM {catalog_name}.{staging_schema}.{table_name.lower()}
        """)
        staging_row_count = staging_df.count()

        #Step 2: Read source count for verification
        source_df = spark.sql(f"""
            SELECT * FROM {source_catalog}.{source_schema}.{table_name}
        """)
        source_row_count = source_df.count()

        #Step 3: Row count check 
        if staging_row_count != source_row_count:
            print(f"{table_name}: Row count mismatch! Source: {source_row_count}, Staging: {staging_row_count}")
            raise Exception(f"Row count mismatch for {table_name}")

        print(f"{table_name}: Row count verified ({staging_row_count} rows)")

        #Step 4: Building dynamic file path
        #execution_time = datetime.now()
        #date_path = execution_time.strftime("%Y/%m/%d")
        #file_location = f"{base_path}/{table_name.lower()}/{date_path}/{table_name.lower()}.parquet"
        execution_time = datetime.now()
        date_path = execution_time.strftime("%Y/%m/%d")
        file_location = f"{base_path}/{table_name.lower()}/{date_path}/{table_name.lower()}"
        print(f"DEBUG file_location: {repr(file_location)}")


        #Step 5: Writing to Raw Volume as Parquet
        #staging_df.write \
        #         .format("parquet") \
        #         .mode("overwrite") \
        #         .save(file_location)
        
        #Step 5: Write to Raw Volume as Delta instead of Parquet (For job pipeline since the version doesn't support parquet write to volume)
        staging_df.write \
                  .format("delta") \
                  .mode("overwrite") \
                  .option("overwriteSchema", "true") \
                  .save(file_location.replace(".parquet", ""))
        

        #Step 6: Verifying target row count
        #target_df = spark.read.format("parquet").load(file_location)
        target_df = spark.read.format("delta").load(file_location)  #for pipeline
        target_row_count = target_df.count()
        status = "SUCCESS" if staging_row_count == target_row_count else "FAILED"

        #Step 7: Logging to child metadata table
        spark.sql(f"""
            INSERT INTO {catalog_name}.{meta_schema}.child_metrics_metadata VALUES (
                '{table_name}',
                '{execution_time}',
                '{status}',
                {source_row_count},
                {target_row_count},
                '{file_location}',
                current_date()
            )
        """)

        print(f"{table_name} | {status} | Source: {source_row_count} | Raw: {target_row_count}")
        print(f"  Written to: {file_location}")

    except Exception as e:
        spark.sql(f"""
            INSERT INTO {catalog_name}.{meta_schema}.child_metrics_metadata VALUES (
                '{table_name}',
                '{datetime.now()}',
                'FAILED',
                0,
                0,
                'N/A',
                current_date()
            )
        """)
        print(f"{table_name} failed: {str(e)}")



Album: Row count verified (347 rows)
Album | SUCCESS | Source: 347 | Raw: 347
  Written to: /Volumes/chinook_catalog/raw_zone/chinook/album/2026/04/02/album.parquet
Artist: Row count verified (275 rows)
Artist | SUCCESS | Source: 275 | Raw: 275
  Written to: /Volumes/chinook_catalog/raw_zone/chinook/artist/2026/04/02/artist.parquet
Customer: Row count verified (62 rows)
Customer | SUCCESS | Source: 62 | Raw: 62
  Written to: /Volumes/chinook_catalog/raw_zone/chinook/customer/2026/04/02/customer.parquet
Employee: Row count verified (8 rows)
Employee | SUCCESS | Source: 8 | Raw: 8
  Written to: /Volumes/chinook_catalog/raw_zone/chinook/employee/2026/04/02/employee.parquet
Genre: Row count verified (25 rows)
Genre | SUCCESS | Source: 25 | Raw: 25
  Written to: /Volumes/chinook_catalog/raw_zone/chinook/genre/2026/04/02/genre.parquet
Invoice: Row count verified (412 rows)
Invoice | SUCCESS | Source: 412 | Raw: 412
  Written to: /Volumes/chinook_catalog/raw_zone/chinook/invoice/2026/04/02/in

In [0]:
file_location = f"{base_path}/{table_name.lower()}/{date_path}/{table_name.lower()}.parquet"
print(file_location)

/Volumes/chinook_catalog/raw_zone/chinook/track/2026/04/02/track.parquet


In [0]:
print(file_location.replace(".parquet", ""))


/Volumes/chinook_catalog/raw_zone/chinook/track/2026/04/02/track
